# 02 — Model Architecture
Kiểm tra cấu trúc `MetaBlock` + `SkinLesionModel` với dummy tensors

## 1. MetaBlock — kiểm tra đơn vị

In [1]:
import sys; sys.path.insert(0, '..')
import torch
from skin_pipeline.utils import MetaBlock, SkinLesionModel

In [2]:
B, visual_dim, meta_dim = 4, 512, 45
mb = MetaBlock(visual_dim, meta_dim)
visual = torch.randn(B, visual_dim)
meta   = torch.randn(B, meta_dim)
out = mb(visual, meta)
print(f'MetaBlock input visual: {visual.shape}')
print(f'MetaBlock input meta:   {meta.shape}')
print(f'MetaBlock output:       {out.shape}  ← phải là [{B}, {visual_dim}] ✓')

MetaBlock input visual: torch.Size([4, 512])
MetaBlock input meta:   torch.Size([4, 45])
MetaBlock output:       torch.Size([4, 512])  ← phải là [4, 512] ✓


## 2. Model chính: PiT + MetaBlock

In [3]:
META_DIM = 45  # được xác định chính xác sau khi chạy Notebook 01
B = 4

model_pit = SkinLesionModel(
    backbone_name='pit_s_distilled_224',
    meta_dim=META_DIM,
    use_metablock=True
)

dummy_img  = torch.randn(B, 3, 224, 224)
dummy_meta = torch.randn(B, META_DIM)

with torch.no_grad():
    logits = model_pit(dummy_img, dummy_meta)

print(f'Backbone: pit_s_distilled_224')
print(f'Visual dim: {model_pit.backbone.num_features}')
print(f'Output logits shape: {logits.shape}  ← phải là [{B}, 6] ✓')

Backbone: pit_s_distilled_224
Visual dim: 576
Output logits shape: torch.Size([4, 6])  ← phải là [4, 6] ✓


## 3. Model Baseline: ResNetBiT + Concat

In [4]:
model_resnet = SkinLesionModel(
    backbone_name='resnetv2_50x1_bit_distilled',
    meta_dim=META_DIM,
    use_metablock=False   # dùng concatenation
)

with torch.no_grad():
    logits2 = model_resnet(dummy_img, dummy_meta)

print(f'Backbone: resnetv2_50x1_bit_distilled')
print(f'Visual dim: {model_resnet.backbone.num_features}')
print(f'Output logits shape: {logits2.shape}  ← phải là [{B}, 6] ✓')

/Users/hhh/workspace/School/PBL7/venv/lib/python3.12/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name resnetv2_50x1_bit_distilled to current resnetv2_50x1_bit.goog_distilled_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Backbone: resnetv2_50x1_bit_distilled
Visual dim: 2048
Output logits shape: torch.Size([4, 6])  ← phải là [4, 6] ✓


## 4. Tóm tắt số tham số (Parameters Count)

In [5]:
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f'PiT + MetaBlock:          {count_params(model_pit):,} parameters')
print(f'ResNetBiT + Concat:       {count_params(model_resnet):,} parameters')

PiT + MetaBlock:          22,964,028 parameters
ResNetBiT + Concat:       23,689,358 parameters
